## Financial Data Preprocessing

A time series is a series of data points indexed in time order. Financial Data such as equity, commodity, and forex price series observed at equally spaced points in time are an example of such a series. It is a sequence of data points observed at regular time intervals and depending on the frequency of observations, a time series may typically be in ticks, seconds, minutes, hourly, daily, weekly, monthly, quarterly and annual.

The first step towards any data analysis would be to parse the raw data that involves extracting the data from the source and then cleaning and filling the missing data if any. While data comes in many forms, Python makes it easy to read time-series data using useful packages.

In this session, we will retrieve and store both end-of-day and intraday data using some of the popular python packages. These libraries aim to keep the API simple and make it easier to access historical data. Further, we will see how to read data from traditional data sources stored locally.

### import libraries

In [ ]:
# data manipuation
import pandas as pd
import numpy as np

# data visualization
import cufflinks as cf                                 # new package - cufflinks
cf.set_config_file(offline=True, theme='pearl')        # cf.set_config()

# Ignore warnings - optional
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf

## Docstring or Signature

In [ ]:
help(yf.download)   # Or Shift + Tab inside funciton parenthesis

## Data Retrival

In [ ]:
# single security
yf.download(tickers = '^NSEI',period = '5y', progress= False)    # yf.download()

# intraday data retrival
yf.download(tickers = '^NSEI',period = '1d',interval='1m', progress= False)    

# multiple securities
tickers_list = ['IBM','^NSEI','BA']

df_tickers_data = yf.download(tickers = tickers_list,period = '1mo')                            # df with multiindex cols
#list_tickers_data = [yf.download(symbol, period = '1mo') for symbol in tickers_list]           # list comprehension
dict_tickers_data = {symbol:yf.download(symbol, period = '1mo') for symbol in tickers_list}     # dictionary comprehension

df_tickers_data['Adj Close']['BA']

In [ ]:
# ticker object
apple= yf.Ticker('AAPL')                                  # yf.Ticker() - will create ticker object

# attributes, writing a few but there are many more
apple.dividends                                          # dividends

# options chain
options = apple.option_chain(date = '2023-12-08')        # option_chain of tiker object
options_call = options.calls                             # call options of ticker object
options_call.loc[options_call['strike'] > 150,:]

## Retrieving Hypertext Markup Language (HTML)

In [ ]:
nifty50 = pd.read_html('https://en.wikipedia.org/wiki/NIFTY_50')[2]   # pd.read_html() - convert tabular data into dataframes

nifty_tickers_list = nifty50['Symbol'].to_list()

## Data Storage

### Storing OHLCV data in Excel File

In [ ]:
writer = pd.ExcelWriter('stocks.xlsx')                                                # pd.ExcelWriter()
[dict_tickers_data[symbol].to_excel(writer,symbol) for symbol in tickers_list]        # to_excel()
writer.save()
writer.close()


### Storing OHLCV data in CSV File

In [ ]:
[dict_tickers_data[symbol].to_csv(symbol+'.csv') for symbol in tickers_list]    # using list comprehension

### Resampling Data

In [ ]:
df_tickers_data['Adj Close'].resample(rule = 'W').last()             # resample()

## Min-Max Index

We'll now use idxmax and idxmin to select the index value that corresponds to the maximum or minimum of the index's percentage change for each week.

In [ ]:
df = df_tickers_data['Adj Close']
df['Week'] = df.index.week                             # week of datetime inde
df.drop(columns = ['BA','IBM'], inplace = True)
df['Return'] = df['^NSEI'].pct_change(periods =1)      # pct_change()

df.groupby('Week')['Return'].idxmax()                  # idxmax()
df.groupby('Week')['Return'].idxmin()                  # idxmin()

## Interactive Visualization of Time Series

We use cufflinks for interactive visualization. It is one of the most feature rich third-party wrapper around Plotly by Santos Jorge. It binds the power of plotly with the flexibility of pandas for easy plotting.

When you import cufflinks library, all pandas data frames and series objects have a new method .iplot() attached to them which is similar to pandas' built-in .plot() method.

In [ ]:
cf.set_config_file(theme='pearl')               # set_config_file()

In [ ]:
df_tickers_data

In [ ]:
df.iplot(kind='line',y = '^NSEI', title='Line Chart')               # line chart

In [ ]:
df_tickers_data.loc[:,('Adj Close',)].loc[:,['BA','IBM']].iplot(kind = 'line',secondary_y='IBM')   # line chart with 2 axes

In [ ]:
df_tickers_data.loc[:,('Adj Close',)].loc[:,['BA','IBM']].iplot(kind = 'line',subplots = True)     # line chart with subplots

In [ ]:
df = df_tickers_data.loc[:,('Adj Close',)].dropna()
df.normalize().iplot()                                               # normalize() 

In [ ]:
df = df_tickers_data.loc[:,('Adj Close',)]
daily_return = np.log(df).diff()                                    # operation on whole dataframe
(daily_return.mean()*252*100).iplot(kind = 'bar')                   # annual return with bar chart

In [ ]:
df = df_tickers_data.loc[:,('Adj Close',)]
daily_return = np.log(df).diff()                                      # operation on whole dataframe
(daily_return.std()*np.sqrt(252)*100).iplot(kind = 'bar')             # annual volatility with bar chart

In [ ]:
df = df_tickers_data.loc[:,('Adj Close',)]
rolling_return = daily_return.rolling(5).sum()           # operation on df
rolling_return.iplot()                                   # rolling return

### Time Series Statistics

Statistics is a branch of mathematics that deals with collecting, interpreting, organization and interpretation of data. The two main categories of statistics are descriptive statistics and inferential statistics.

Descriptive statistics help us to understand the data in a meaningful way and is an important part of data analysis. While inferential statistics allows us to infer trends and derive conclusion from it.

### Log Normal Distribution

A normal distribution is the most common and widely used distribution in statistics. It is popularly referred as a “bell curve” or “Gaussian curve”. Financial time series though random in short term, follows a log normal distribution on a longer time frame.

Now that we have derived the daily log returns, we will plot this return distribution and check whether the stock returns follows log normality.

In [ ]:
daily_return.iplot(kind = 'histogram', subplots= True, title = 'Histogram of Daily Returns',bins=5)  # histogram

In [ ]:
daily_return.corr().iplot(kind='heatmap', title="Correlation Matrix", colorscale="Blues")       # correlation matrix

## References

[YFinance Documentation](https://github.com/ranaroussi/yfinance)

[Cufflinks Documentation](https://github.com/santosjorge/cufflinks)

[Numpy Documentation](https://numpy.org/doc/stable/)

[Pandas Documentation](https://pandas.pydata.org/)

[Python Resources](https://github.com/kannansingaravelu/PythonResources)

[SQLAlchemy Documentation](https://docs.sqlalchemy.org/en/20/)

[SQLLite Documentation](https://www.sqlite.org/docs.html)
